# Nova AI — Colab Trainer 🚀

Train the Nova models (Nova 1.9 + Supernova 1.0) **longer and faster** than the sandbox can, straight from this notebook.

**How to use (3 steps):**
1. `Runtime → Run all` (or press the play buttons top to bottom)
2. Wait — with a GPU it's roughly 10-20x faster per step; without one it still trains, just slower
3. When the last cell finishes, the two trained checkpoint files download automatically — **send both files back to Solene** (upload them in the chat) and she verifies + ships them to the live site

Optional: set `EXTRA_TURNS` in the data cell to teach Nova extra things (format: `You: question\nNova: answer` blocks separated by blank lines).

Everything stays in the Phollama-OS family: same code, same corpus, just more compute.

In [ ]:
#@title 1. Setup: Node.js + repo + TensorFlow (auto-detects GPU)
import subprocess, shutil, os

gpu = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
print('GPU available:', gpu, '(Runtime → Change runtime type → T4 GPU gives a big speedup)')

!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs > /dev/null 2>&1
print('Node:', subprocess.run(['node', '--version'], capture_output=True).text.strip())

if not os.path.exists('Phollama-OS'):
    !git clone -q https://github.com/genity5-collab/Phollama-OS.git
%cd Phollama-OS
!git pull -q 2>/dev/null

# tfjs-node (CPU) always installed; if a GPU is present we try the GPU build via an alias,
# and fall back silently to CPU if the GPU build misbehaves.
!npm install --no-fund --no-audit --silent @tensorflow/tfjs-core@4.22.0 @tensorflow/tfjs-node@4.22.0 > /dev/null 2>&1
if gpu:
    !npm install --no-fund --no-audit --silent @tensorflow/tfjs-node@npm:@tensorflow/tfjs-node-gpu@4.22.0 --force > /dev/null 2>&1
    !npm install --no-fund --no-audit --silent @tensorflow/tfjs-node-gpu@4.22.0 > /dev/null 2>&1

# verify the trainer actually starts (prints a version line, exits 0)
t = subprocess.run(['node', '-e', "require('@tensorflow/tfjs-node'); console.log('tfjs-node OK, backend:', require('@tensorflow/tfjs-node').getBackend ? 'loaded' : 'loaded')"], capture_output=True, text=True)
print(t.stdout or t.stderr)

In [ ]:
#@title 2. Data: pull the latest corpus + optionally add your own turns
!git pull -q 2>/dev/null

EXTRA_TURNS = "" #@param {type:"string"}

with open('train/corpus.txt', 'a') as f:
    if EXTRA_TURNS.strip():
        f.write('\n' + EXTRA_TURNS.strip() + '\n')
        print('Added', len(EXTRA_TURNS), 'chars of extra turns')

corpus = open('train/corpus.txt').read()
turns = corpus.count('You: ')
print(f'Corpus ready: {len(corpus):,} chars, ~{turns} training turns')

In [ ]:
#@title 3. Train both models (this is the long cell — go get a snack)
EPOCHS = 100 #@param {type:"integer"}
MAX_SEQS = 5000 #@param {type:"integer"}

import time
for model in ['nova-1.9', 'supernova-1.0']:
    print(f'\n========== training {model} ({EPOCHS} epochs x {MAX_SEQS} seqs) ==========')
    t0 = time.time()
    !MODEL={model} EPOCHS={EPOCHS} MAX_SEQS={MAX_SEQS} node train/train.js
    print(f'>> {model} finished in {(time.time()-t0)/60:.1f} min')

In [ ]:
#@title 4. Download the two trained checkpoints
from google.colab import files
print('Download these two files, then upload both to Solene in the chat — she tests and ships them.')
files.download('js/base-checkpoints/nova-1.9.js')
files.download('js/base-checkpoints/supernova-1.0.js')

**Tip:** a free Colab session can run ~12 hours. Crank `EPOCHS` up to 200-300 for a seriously long training run — every extra epoch lowers the loss and sharpens answers. Just keep the browser tab open.

Batch-size note: `MAX_SEQS` controls how many training samples each epoch sees. 5000 is a good default; bigger = smoother but slightly slower per epoch.